In [1]:
# Importando bibliotecas
import pandas as pd


### 01 — Análise Inicial do Dataset



#### Objetivo

Realizar auditoria inicial do dataset laboratorial para:

- entender a estrutura dos dados
- identificar colunas críticas
- avaliar qualidade dos dados
- identificar possíveis dimensões
- **mapear colunas sensíveis**
- compreender a granularidade da tabela

#### Contexto

Os dados foram extraídos diretamente do sistema laboratorial e serão utilizados para construção de um pipeline analítico com Python + Power BI.

In [2]:
# Importando os dados

df = pd.read_excel('../data/raw/01.01.2025.xlsx')

In [3]:
# Dimensoes do dataset
df.shape

(22805, 76)

#### Estrutura incial do dataset

O dataset tem:
- 22805 linhas
- 76 colunas

A leitura inicial foi feita diretamente no arquivo Excel bruto.

In [4]:
# Verificando colunas
df.columns.tolist()

['Unnamed: 0',
 'parametroDataInicial',
 'parametroDataFinal',
 'parametroEmpresa',
 'parametroProcedimentos',
 'parametroOrigens',
 'ParametroUsuarioLogado',
 'NumeroPedido',
 'PrescricaoMedica',
 'ProntuárioHIAEGO',
 'RegistroExternoCerner',
 'Amostra',
 'ContadorAmostra',
 'CodigoMatrix',
 'CodigoFaturamento',
 'NomeProc',
 'CodigoInstrumento',
 'LoteTriagem',
 'Material',
 'MATERIALCERNER',
 'RegiaoColeta',
 'Meio',
 'MeioCERNER',
 'Volume',
 'UnidadeProdutiva',
 'Area',
 'NomeArea',
 'GrupoInterface',
 'GrupoOrigem',
 'NomeGrupoOrigem',
 'Origem',
 'NomeOrigem',
 'Clinica',
 'NomeClinica',
 'Usuario',
 'DataSistema',
 'DataHoraSolicitacaoMedica',
 'DataHoraPedido',
 'TipoInclusaoExame',
 'Tempointegracao',
 'DataCriacaoAmostra',
 'HoraCriacaoAmostra',
 'DataHoraImpressao',
 'DataHoraColeta',
 'DataHoraCheckout',
 'DataHoraSaidaApoiado',
 'DataHoraTriagem',
 'DataHoraRecepcaoAreaTecnica',
 'DataHoraResultado',
 'ResponsavelLiberacaoTecnica',
 'DataHoraLiberacaoTecnica',
 'Responsav

In [5]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 22805 entries, 0 to 22804
Data columns (total 76 columns):
 #   Column                          Non-Null Count  Dtype         
---  ------                          --------------  -----         
 0   Unnamed: 0                      0 non-null      float64       
 1   parametroDataInicial            22805 non-null  datetime64[us]
 2   parametroDataFinal              22805 non-null  datetime64[us]
 3   parametroEmpresa                22805 non-null  int64         
 4   parametroProcedimentos          0 non-null      float64       
 5   parametroOrigens                0 non-null      float64       
 6   ParametroUsuarioLogado          22805 non-null  str           
 7   NumeroPedido                    22805 non-null  int64         
 8   PrescricaoMedica                22793 non-null  float64       
 9   ProntuárioHIAEGO                0 non-null      float64       
 10  RegistroExternoCerner           18114 non-null  float64       
 11  Amostra      

#### Primeiras observações estruturais

- 3 colunas datetime
- Presença de dados sensívis, como nome, CPF, alguns registros de prontuários
- Presença de valores ausentes
- colunas com dados operacionais

#### Possíveis dimensões


- área
- Origem
- Material
- Exame

- Existem possíveis colunas sensíveis que precisarão ser removidas ou anonimizadas.



In [6]:
# Validando pedidos unicos
df['NumeroPedido'].nunique()

#A intenção aqui é verificar se cada linha do dataset representa um exame ou um pedido

4075

In [7]:
df.shape[0]

22805

In [8]:
df[['NumeroPedido', 'CodigoMatrix', 'NomeProc']].head(20)

,NumeroPedido,CodigoMatrix,NomeProc
0,141820110,HEM,HEMOGRAMA COM PLAQUETAS
1,141820110,UREIA,DOSAGEM DE UREIA
2,141820110,CREAT,DOSAGEM DE CREATININA
3,141820110,TGO,DOSAGEM DE TGO - AST - ASPARTATO AMINOTRANSF...
4,141820110,TGP,DOSAGEM DE TGP - ALT - ALANINA AMINOTRANSFERASE
5,141820110,GGT,DOSAGEM DE GAMA GT
6,141820110,AMI,DOSAGEM DE AMILASE
7,141820110,URITIURN,URINA TIPO I - URINA
8,141820128,GAA,GASOMETRIA ARTERIAL
9,141820152,BETAHCGU,BETA HCG NA URINA QUALITATIVO - URINA


**Conclusão -**
Cada linha do dataset representa um exame laboratorial individual vinculado a um pedido médico.

Um único pedido pode conter múltiplos exames laboratoriais.

### 01 — Análise Exploratória

In [9]:
nulls = (df.isnull().sum())
        
nulls

Unnamed: 0                22805
parametroDataInicial          0
parametroDataFinal            0
parametroEmpresa              0
parametroProcedimentos    22805
                          ...  
CPF                          13
DadosTecnicos             22805
Observacao2               22805
StatusProducao                0
DataHoraRetiradaProc          0
Length: 76, dtype: int64

#### Remoção de colunas totalmente nulas

Foram identificadas colunas completamente vazias no dataset original.

Essas colunas não possuem valor analítico nem operacional e serão removidas para otimização do dataset.

In [10]:
colunas_remover = [
    'Unnamed: 0',
    'parametroProcedimentos',
    'parametroOrigens',
    'ProntuárioHIAEGO',
    'CodigoInstrumento',
    'DataHoraSolicitacaoMedica',
    'Tempointegracao',
    'DataHoraSaidaApoiado',
    'DataHoraRecepcaoAreaTecnica',
    'DadosTecnicos',
    'Observacao2',
    'RegiaoColeta',
    'DataHoraCheckout',
    'CodigoMPP',
    'DescricaoMPP',
    'ResponsavelLiberacaoTecnica',
    'ResponsavelLiberacaoClinica',
    'UsuarioCancelaLiberacao',
    'DataLiberacaoCancelada',
    'MATERIALCERNER',
    'Volume',
    'StatusProducao',
    'DataNascimento',
    'FlagMPP',
    'FlagParcial',
    'GrupoInterface',
    'Volume',
    'MeioCERNER',
    'ParametroUsuarioLogado',
    'DataHoraCancelaLiberacao',
    'NomePaciente',
    'CPF',
    'CRM',
    'MedicoSolicitante',
    'Usuario',
    'RegistroExternoCerner',
    'Laudo',
    'PrescricaoMedica',
    'Registro'
]

df = df.drop(columns=colunas_remover)

In [11]:
df.shape

(22805, 38)

In [12]:
# Verificando colunas após remoção de colunas desnecessárias
df.columns.to_list()

['parametroDataInicial',
 'parametroDataFinal',
 'parametroEmpresa',
 'NumeroPedido',
 'Amostra',
 'ContadorAmostra',
 'CodigoMatrix',
 'CodigoFaturamento',
 'NomeProc',
 'LoteTriagem',
 'Material',
 'Meio',
 'UnidadeProdutiva',
 'Area',
 'NomeArea',
 'GrupoOrigem',
 'NomeGrupoOrigem',
 'Origem',
 'NomeOrigem',
 'Clinica',
 'NomeClinica',
 'DataSistema',
 'DataHoraPedido',
 'TipoInclusaoExame',
 'DataCriacaoAmostra',
 'HoraCriacaoAmostra',
 'DataHoraImpressao',
 'DataHoraColeta',
 'DataHoraTriagem',
 'DataHoraResultado',
 'DataHoraLiberacaoTecnica',
 'DataHoraLiberacaoClinica',
 'FlagLiberacaoClinicaAutomatica',
 'ResultadoRetificado',
 'DataHoraCancelamento',
 'Recoleta',
 'Sexo',
 'DataHoraRetiradaProc']

In [13]:
# Exportando o dataset
df.to_csv(
    '../data/processed/dataset_v1.csv',
    index=False
)